In [1]:
from google.colab import userdata
!pip install langchain-openai
import langchain
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.8 MB/s eta 0:00:00


In [2]:
#@title Install Packages and Setup Environment { display-mode: "form" }

from google.colab import files
import zipfile
import io
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# potential solution
!pip install protobuf==3.20 &> /dev/null
!pip install tensorflow==2.8 &> /dev/null
!apt install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2 &> /dev/null


!pip -q install streamlit &> /dev/null
!pip -q install pyngrok &> /dev/null
from pyngrok import ngrok

#not necessary for now
#!pip install tensorflowjs
#import tensorflowjs as tfjs

import json
from urllib.request import urlretrieve

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Image
import os
import gdown

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

def capture_image(filename):
  try:
    take_photo(filename)
    print('Saved to {}'.format(filename))

    # Show the image which was just taken.
    display(Image(filename))
  except Exception as err:
    # Errors will be thrown if the user does not have a webcam or if they do not
    # grant the page permission to access it.
    print(str(err))

import numpy as np
import tensorflow as tf
import cv2
from google.colab.patches import cv2_imshow

# Ensure comptability with different TF versions
version_fn = getattr(tf.keras, "version", None)
if version_fn and version_fn().startswith("3."):
  import tf_keras as keras
else:
  keras = tf.keras

import warnings
warnings.filterwarnings('ignore')


In [3]:
#@title Upload and extract model into Colab

# upload model zip file
uploaded = files.upload()

# extract model
try:
  file_name = list(uploaded.keys())[-1]
  print("Extracting model...")
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
      zip_ref.extractall('')

  class_names = np.genfromtxt("labels.txt", dtype="str", delimiter='\n')
  for i in range(len(class_names)):
    class_names[i] = ' '.join(class_names[i].split(' ')[1::])

  predictions = [0.0] * len(class_names)

  model = keras.models.load_model('keras_model.h5', compile=False)
  print("Success! Model Extracted!")
except (IndexError, NameError):
  print("Oops! Cannot find file to unzip. Please try uploading a zip file for your model")

Saving converted_keras (1).zip to converted_keras (1).zip
Extracting model...
Success! Model Extracted!


In [4]:
 ok_model=ChatOpenAI(
    api_key=userdata.get('Franc'),
    base_url='https://open.bigmodel.cn/api/paas/v4',
    model='glm-4',
    max_completion_tokens=100
  )


In [5]:
embeddings= OpenAIEmbeddings(
    api_key=userdata.get("Franc"),
    base_url="https://open.bigmodel.cn/api/paas/v4",
    model="embedding-3",
)

In [6]:
prompt2= """Yuu are a helpfull assistant for farmers,
 you will be receiving a name of the maize disease and use the given context to answer how does the disease affect
 and the ways to avoid it from maize.Provide only essential symptoms and treatment recommendations for {Disease} in maize. Keep it concise.
 ***Disease***
  {Disease}
 ***Answer***

 When you recieve the disease you will be answering like this;
 Disease_name:
 Effects:
 Solution:

 """

In [7]:
prompts="""Answer the following questions as best you can. You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: Analyze the image of a maize leaf showing signs of disease. Identify the specific disease affecting the maize plant, and provide a brief description of its symptoms. Then, recommend effective treatment or prevention methods suitable for smallholder farmers. Respond clearly and concisely.
{agent_scratchpad}
"""

In [8]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool

In [9]:
import os
# image_path = "maize_leaf_image.jpg"
# print("Full image path:", os.path.abspath(image_path))
# img = cv2.imread(os.path.abspath(image_path))

In [ ]:

import os
import cv2
import numpy as np
import tensorflow as tf
import gradio as gr
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool

# Tools
@tool
def Search_for_disease_details(predicted_class_name):
    """Returns essential info about the maize disease."""
    prompt_template = PromptTemplate.from_template(template=prompt2)
    chain = LLMChain(llm=ok_model, prompt=prompt_template)
    resp = chain.invoke(input={'Disease': predicted_class_name})
    return resp['text']


@tool
def Find_disease(image_path):
    """Analyzes a maize leaf image to identify disease and return key info."""

    img = cv2.imread(image_path)
    # if img is None:
    #     raise ValueError(f"❌ Could not read image at: {image_path}. Ensure it's a valid format.")

    img = cv2.resize(img, (224, 224))
    img = np.expand_dims(img, axis=0)
    img = img.astype(np.float32) / 255.0

    predictions = model.predict(img)
    score = tf.nn.softmax(predictions[0])

    predicted_class_index = np.argmax(score)
    predicted_class_name = class_names[predicted_class_index]
    confidence = float(np.max(score))

    print(f"✅ Predicted: {predicted_class_name} ({confidence*100:.2f}% confidence)")

    disease_info = Search_for_disease_details(predicted_class_name)
    return disease_info


tools = [Find_disease, Search_for_disease_details]
prot = PromptTemplate.from_template(prompts)
agent = create_react_agent(ok_model, tools=tools, prompt=prot)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)


def process_image(image_path):
    print(f"🧾 Received image path: {image_path}")
    response = executor.invoke({"input": image_path})
    return response


def ui():
    chat_ui = gr.Interface(
        fn=process_image,
        inputs=gr.Image(type="filepath"),
        outputs=gr.TextArea(label="Disease Information"),
        title="GOOD MAIZE AI",
        description="Upload a maize leaf image to detect disease and get actionable insights 🌿🧠"
    )
    chat_ui.launch(debug=True)


if __name__ == "__main__":
    ui()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://02b8d4c6c309f7bee8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🧾 Received image path: /tmp/gradio/13e32640b9606f686eb6b987cf22c630cd3f63a41d5be63c52afd888db094d73/imgi_11_default.jpg


> Entering new AgentExecutor chain...
Thought: I need to analyze the image of the maize leaf to identify the disease and then find more details about it.
Action: Find_disease
Action Input: image_path (path to the image file)
Observation

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 626, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 350, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2235, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1746, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^